In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from matplotlib.mathtext import _mathtext as mathtext
mathtext.FontConstantsBase.sup1 = 0.45
import matplotlib.colors as mcolors
import cartopy.crs as ccrs

# 数据加载
test_set_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/test_set.pkl')
importance_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/importance.pkl').sort_values('Importance', ascending=False)
map_df = pd.read_csv('../../data/main_figures/model_results/AGB_upscale_2020_GB_1pctdeg.csv')

# 子图a：散点图
observed_agb = test_set_df['Observed_AGB']
estimated_agb = test_set_df['Estimated_AGB']
rmse = np.sqrt(mean_squared_error(observed_agb, estimated_agb))
r2 = r2_score(observed_agb, estimated_agb)
coefficients = np.polyfit(observed_agb, estimated_agb, 1)
polynomial = np.poly1d(coefficients)
equation = f"y = {polynomial.coefficients[0]:.2f}x + {polynomial.coefficients[1]:.2f}\nR$^2$ = {r2:.2f}\nRMSE = {rmse:.0f} Mg DM ha$^{{-1}}$"

# 子图b：重要性图
features = importance_df['Feature']
importance = importance_df['Importance']
importance_pct = (importance / importance.sum()) * 100

categories = {
    'Climatic': ['Tavg', 'Tmin', 'Pavg', 'Pmin', 'Srad', 'VPD', 'PET', 'Cyclone'],
    'Hydrologic': ['Runoff', 'TidalRange', 'Dist2Coast', 'Salinity', 'SeaTemp'],
    'Topographic': ['DEM', 'Slope'],
    'Anthropogenic': ['Pop'],
    'Height': ['Height']
}

color_map = {
    'Climatic': '#FFF1DC',
    'Hydrologic': '#D1F1FE',
    'Topographic': '#E6E6E6',
    'Anthropogenic': '#EBDEF6',
    'Height': '#C1EDDE'
}
feature_to_category = {feature: cat for cat, feats in categories.items() for feature in feats}
colors = [mcolors.hex2color(color_map[feature_to_category[feature]]) if feature in feature_to_category else 'black' for feature in features]

# 调整颜色亮度
def brighten_rgb(color, beta):
    """
    Brighten or darken a single RGB color.
    
    Parameters:
    color (tuple): A tuple representing an RGB color, e.g., (0.5, 0.2, 0.7)
    beta (float): A value to control the brightness. If 0 < beta <= 1, the color becomes brighter,
                  and if -1 <= beta < 0, it becomes darker.

    Returns:
    tuple: The adjusted RGB color.
    """
    color = np.clip(color, 0, 1)
    if beta > 0:
        gamma = 1 - min(1 - np.sqrt(np.finfo(float).eps), beta)
    else:
        gamma = 1 / (1 + max(-1 + np.sqrt(np.finfo(float).eps), beta))
    return tuple(np.clip(np.array(color) ** gamma, 0, 1))

edge_colors = [brighten_rgb(color, -0.9) for color in colors]  # 应用暗化边框颜色

# Set global font to Helvetica
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
# Set the mathtext font to Helvetica
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'


# 创建布局
fig = plt.figure(figsize=(12, 9), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=[1, 0.5], width_ratios=[1, 1])

# 子图a：散点图
ax1 = fig.add_subplot(grid[0, 0])
ax1.scatter(observed_agb, estimated_agb, facecolors='none', edgecolors='gray', s=50, label="Data points")
plot_range = [-50, 650]
ax1.plot(plot_range, plot_range, linestyle='--', color='gray', label="1:1 line")
ax1.plot(plot_range, polynomial(plot_range), color='red', label="Regression line")
ax1.set_xlim(plot_range)
ax1.set_ylim(plot_range)
ax1.set_xlabel("Observed AGB density (Mg DM ha$^{-1}$)", labelpad=0)
ax1.set_ylabel("Estimated AGB density (Mg DM ha$^{-1}$)")
ax1.text(0, 450, equation)
ax1.set_title('a', loc='left', fontweight='bold')

# 子图b：柱状图
ax2 = fig.add_subplot(grid[0, 1])
ax2.barh(features, importance_pct, color=colors, edgecolor=edge_colors, height=0.6, linewidth=1.5)  # 添加暗化边框
ax2.set_xlabel("Importance (%)", labelpad=10)
ax2.invert_yaxis()
ax2.set_title('b', loc='left', fontweight='bold')

# 添加饼图
pie_ax = fig.add_axes([0.70, 0.45, 0.22, 0.22])  # 控制饼图位置和大小
category_percentages = {
    cat: importance_pct[features.isin(feats)].sum() for cat, feats in categories.items()
}
pie_colors = [color_map[cat] for cat in category_percentages.keys()]
pie_edge_colors = [brighten_rgb(mcolors.hex2color(color), -0.8) for color in pie_colors]
explode = [0.1] * len(category_percentages)

wedges, texts, autotexts = pie_ax.pie(
    category_percentages.values(),
    labels=category_percentages.keys(),
    colors=pie_colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.5},
    textprops={'fontsize': 14}
)

# 为饼图边框添加暗化颜色
for wedge, edge_color in zip(wedges, pie_edge_colors):
    wedge.set_edgecolor(edge_color)

# 子图c：地图
ax3 = fig.add_subplot(grid[1, :], projection=ccrs.PlateCarree())
ax3.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=0)
ax3.add_feature(ccrs.cartopy.feature.LAND, facecolor='white', zorder=0)
ax3.set_extent([-180, 180, -40, 40], crs=ccrs.PlateCarree())
scatter = ax3.scatter(map_df['Lon'], map_df['Lat'], c=map_df['AGB'], cmap='YlGn', s=5, edgecolor='none', transform=ccrs.PlateCarree(), zorder=1)
scatter.set_clim(0, 301)
ax3.set_title('c', loc='left', fontweight='bold')

# 添加色带
cbar_ax = fig.add_axes([0.135, 0.06, 0.15, 0.02])  # 控制色带位置和大小
cbar = plt.colorbar(scatter, cax=cbar_ax, orientation='horizontal')
cbar.ax.set_title('Mangrove AGB density (Mg DM ha$^{-1}$)', fontsize=14, pad=10)
cbar.set_ticks([0, 100, 200, 300])
cbar.ax.tick_params(labelsize=14)

# 保存并展示图形
fig.savefig('py01_machine-learning_AGB.png', dpi=600, transparent=True)
plt.show()

In [ ]:
selected_features = ['Pavg', 'VPD', 'PET', 'Cyclone', 'Srad', 'Salinity', 'SeaTemp', 'Tavg', 'Runoff', 'Pop', 'Tmin', 'Pmin']
total_importance_pct = importance_pct[features.isin(selected_features)].sum()
print(f"总重要性占比之和为: {total_importance_pct:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Serif SC'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Noto Serif SC'
plt.rcParams['mathtext.it'] = 'Noto Serif SC:italic'
plt.rcParams['mathtext.bf'] = 'Noto Serif SC:bold'


import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
# --- 数据加载 ---
test_set_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/test_set.pkl')

# --- 数据计算 ---
observed_agb = test_set_df['Observed_AGB']
estimated_agb = test_set_df['Estimated_AGB']
rmse = np.sqrt(mean_squared_error(observed_agb, estimated_agb))
r2 = r2_score(observed_agb, estimated_agb)
coefficients = np.polyfit(observed_agb, estimated_agb, 1)
polynomial = np.poly1d(coefficients)
# 使用中文标签
equation = f"y = {polynomial.coefficients[0]:.2f}x + {polynomial.coefficients[1]:.2f}\nR$^2$ = {r2:.2f}\nRMSE = {rmse:.0f} Mg DM ha$^{{-1}}$"

# --- 开始绘图 ---
fig, ax = plt.subplots(figsize=(6, 6))

# 绘制散点
ax.scatter(observed_agb, estimated_agb, facecolors='none', edgecolors='gray', s=50, label="数据点")

# 设定绘图范围并绘制1:1线和回归线
plot_range = [-50, 650]
ax.plot(plot_range, plot_range, linestyle='--', color='gray', label="1:1线")
ax.plot(observed_agb, polynomial(observed_agb), color='red', label="回归线")

# 设置坐标轴
ax.set_xlim(plot_range)
ax.set_ylim(plot_range)
ax.set_xlabel("观测的AGB密度 (Mg DM ha$^{-1}$)")
ax.set_ylabel("估算的AGB密度 (Mg DM ha$^{-1}$)")

# 添加公式文本
ax.text(0.05, 0.95, equation, transform=ax.transAxes, verticalalignment='top')

# 调整布局并显示
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- 1. 恢复用于调整颜色亮度的函数 ---
def brighten_rgb(color, beta):
    color = np.clip(color, 0, 1)
    if beta > 0:
        gamma = 1 - min(1 - np.sqrt(np.finfo(float).eps), beta)
    else:
        gamma = 1 / (1 + max(-1 + np.sqrt(np.finfo(float).eps), beta))
    return tuple(np.clip(np.array(color) ** gamma, 0, 1))

# --- 数据加载 ---
importance_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/importance.pkl').sort_values('Importance', ascending=False)

# --- 数据准备与翻译 ---
feature_translation = {
    'Height': '树冠高度', 'Pavg': '年降水量', 'TidalRange': '潮差',
    'Dist2Coast': '距海岸距离', 'VPD': '饱和水汽压差', 'PET': '潜在蒸散量',
    'Cyclone': '气旋频率', 'Srad': '太阳辐射', 'Salinity': '海水盐度',
    'SeaTemp': '海面温度', 'Tavg': '年平均温度', 'Runoff': '径流量',
    'Pop': '人口密度', 'Tmin': '最冷月最低温度', 'Pmin': '最干旱月降水量', 'DEM': '海拔',
    'Slope': '坡度'
}
importance_df['Feature_CN'] = importance_df['Feature'].map(feature_translation)
features_cn = importance_df['Feature_CN']
importance_pct = (importance_df['Importance'] / importance_df['Importance'].sum()) * 100

categories = {
    '气候': ['年平均温度', '最冷月最低温度', '年降水量', '最干旱月降水量', '太阳辐射', '饱和水汽压差', '潜在蒸散量', '气旋频率'],
    '水文': ['径流量', '潮差', '距海岸距离', '海水盐度', '海面温度'],
    '地形': ['海拔', '坡度'],
    '人为': ['人口密度'],
    '林分结构': ['树冠高度']
}
color_map = {
    '气候': '#FFF1DC', '水文': '#D1F1FE', '地形': '#E6E6E6',
    '人为': '#EBDEF6', '林分结构': '#C1EDDE'
}

feature_to_category_cn = {feat_cn: cat_cn for cat_cn, feats_cn in categories.items() for feat_cn in feats_cn}
# 将十六进制颜色码转为RGB元组
colors_rgb = [mcolors.hex2color(color_map.get(feature_to_category_cn.get(feat), 'gray')) for feat in features_cn]
# --- 2. 使用 brighten_rgb 函数生成深色边框 ---
edge_colors = [brighten_rgb(color, -0.9) for color in colors_rgb]

# --- 开始绘图 ---
fig, ax = plt.subplots(figsize=(6, 6))

# 绘制柱状图 (使用恢复的颜色)
ax.barh(features_cn, importance_pct, color=colors_rgb, edgecolor=edge_colors, height=0.7, linewidth=1.5) # <--- 修改此处
ax.set_xlabel("重要性 (%)", labelpad=10)
ax.invert_yaxis()

# --- 添加饼图 ---
pie_ax = fig.add_axes([0.5, 0.25, 0.4, 0.4])
category_percentages = {}
for cat_cn, feats_cn in categories.items():
    is_in_category = importance_df['Feature_CN'].isin(feats_cn)
    category_percentages[cat_cn] = importance_pct[is_in_category].sum()

pie_colors_hex = [color_map[cat] for cat in category_percentages.keys()]
pie_edge_colors = [brighten_rgb(mcolors.hex2color(color), -0.8) for color in pie_colors_hex]
# --- 3. 恢复饼图的 "explode" 效果 ---
explode = [0.1] * len(category_percentages) # <--- 恢复此行

wedges, texts, autotexts = pie_ax.pie(
    category_percentages.values(),
    labels=category_percentages.keys(),
    colors=pie_colors_hex,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode, # <--- 应用 "explode"
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2},
    textprops={'fontsize': 14}
)

# 恢复饼图的深色边框
for wedge, edge_color in zip(wedges, pie_edge_colors):
    wedge.set_edgecolor(edge_color)

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import mean_squared_error, r2_score

# --- 全局字体和样式设置 ---
plt.rcParams['font.family'] = 'Noto Serif SC'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Noto Serif SC'
plt.rcParams['mathtext.it'] = 'Noto Serif SC:italic'
plt.rcParams['mathtext.bf'] = 'Noto Serif SC:bold'

# --- 辅助函数定义 ---
def brighten_rgb(color, beta):
    color = np.clip(color, 0, 1)
    if beta > 0:
        gamma = 1 - min(1 - np.sqrt(np.finfo(float).eps), beta)
    else:
        gamma = 1 / (1 + max(-1 + np.sqrt(np.finfo(float).eps), beta))
    return tuple(np.clip(np.array(color) ** gamma, 0, 1))

# --- 数据加载 ---
test_set_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/test_set.pkl')
importance_df = pd.read_pickle('../../data/main_figures/model_output/GB_WorldClim_Potapov/importance.pkl').sort_values('Importance', ascending=False)


# --- 1. 准备散点图数据 ---
observed_agb = test_set_df['Observed_AGB']
estimated_agb = test_set_df['Estimated_AGB']
rmse = np.sqrt(mean_squared_error(observed_agb, estimated_agb))
r2 = r2_score(observed_agb, estimated_agb)
coefficients = np.polyfit(observed_agb, estimated_agb, 1)
polynomial = np.poly1d(coefficients)
equation = f"y = {polynomial.coefficients[0]:.2f}x + {polynomial.coefficients[1]:.2f}\nR$^2$ = {r2:.2f}\nRMSE = {rmse:.0f} Mg DM ha$^{{-1}}$"


# --- 2. 准备重要性图数据 ---
feature_translation = {
    'Height': '树冠高度', 'Pavg': '年降水量', 'TidalRange': '潮差',
    'Dist2Coast': '距海岸距离', 'VPD': '饱和水汽压差', 'PET': '潜在蒸散量',
    'Cyclone': '气旋频率', 'Srad': '太阳辐射', 'Salinity': '海水盐度',
    'SeaTemp': '海面温度', 'Tavg': '年平均温度', 'Runoff': '径流量',
    'Pop': '人口密度', 'Tmin': '最冷月最低温度', 'Pmin': '最干旱月降水量', 'DEM': '海拔',
    'Slope': '坡度'
}
importance_df['Feature_CN'] = importance_df['Feature'].map(feature_translation)
features_cn = importance_df['Feature_CN']
importance_pct = (importance_df['Importance'] / importance_df['Importance'].sum()) * 100

categories = {
    '气候': ['年平均温度', '最冷月最低温度', '年降水量', '最干旱月降水量', '太阳辐射', '饱和水汽压差', '潜在蒸散量', '气旋频率'],
    '水文': ['径流量', '潮差', '距海岸距离', '海水盐度', '海面温度'],
    '地形': ['海拔', '坡度'],
    '人为': ['人口密度'],
    '林分结构': ['树冠高度']
}
color_map = {
    '气候': '#FFF1DC', '水文': '#D1F1FE', '地形': '#E6E6E6',
    '人为': '#EBDEF6', '林分结构': '#C1EDDE'
}

feature_to_category_cn = {feat_cn: cat_cn for cat_cn, feats_cn in categories.items() for feat_cn in feats_cn}
colors_rgb = [mcolors.hex2color(color_map.get(feature_to_category_cn.get(feat), 'gray')) for feat in features_cn]
edge_colors = [brighten_rgb(color, -0.9) for color in colors_rgb]


# --- 开始绘图：创建一个1行2列的画布 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# --- 绘制左图 (ax1): 散点图 ---
ax1.scatter(observed_agb, estimated_agb, facecolors='none', edgecolors='gray', s=50, label="数据点")
plot_range = [-50, 650]
ax1.plot(plot_range, plot_range, linestyle='--', color='gray', label="1:1线")
ax1.plot(observed_agb, polynomial(observed_agb), color='red', label="回归线")
ax1.set_xlim(plot_range)
ax1.set_ylim(plot_range)
ax1.set_xlabel("观测的AGB密度 (Mg DM ha$^{-1}$)")
ax1.set_ylabel("估算的AGB密度 (Mg DM ha$^{-1}$)")
ax1.text(0.05, 0.95, equation, transform=ax1.transAxes, verticalalignment='top')
ax1.set_title('a', loc='left', fontweight='bold')


# --- 绘制右图 (ax2): 重要性柱状图 ---
ax2.barh(features_cn, importance_pct, color=colors_rgb, edgecolor=edge_colors, height=0.7, linewidth=1.5)
ax2.set_xlabel("重要性 (%)", labelpad=10)
ax2.invert_yaxis()
ax2.set_title('b', loc='left', fontweight='bold')


# --- 在右图上添加内嵌饼图 ---
# 关键：重新计算饼图位置 [left, bottom, width, height]，坐标是相对于整个fig而言的
pie_ax = fig.add_axes([0.65, 0.2, 0.35, 0.35])
category_percentages = {}
for cat_cn, feats_cn in categories.items():
    is_in_category = importance_df['Feature_CN'].isin(feats_cn)
    category_percentages[cat_cn] = importance_pct[is_in_category].sum()

pie_colors_hex = [color_map[cat] for cat in category_percentages.keys()]
pie_edge_colors = [brighten_rgb(mcolors.hex2color(color), -0.8) for color in pie_colors_hex]
explode = [0.1] * len(category_percentages)

wedges, texts, autotexts = pie_ax.pie(
    category_percentages.values(),
    labels=category_percentages.keys(),
    colors=pie_colors_hex,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2},
    textprops={'fontsize': 14} # 缩小一点字体以适应布局
)
for wedge, edge_color in zip(wedges, pie_edge_colors):
    wedge.set_edgecolor(edge_color)

# --- 最终调整与显示 ---
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- 数据加载 ---
map_df = pd.read_csv('../../data/main_figures/model_results/AGB_upscale_2020_GB_1pctdeg.csv')

# --- 开始绘图 ---
fig, ax = plt.subplots(figsize=(12, 3), subplot_kw={'projection': ccrs.PlateCarree()})

# 设置地图范围
ax.set_extent([-180, 180, -40, 40], crs=ccrs.PlateCarree())

# 添加白色陆地背景，并设置其为最底层
ax.add_feature(cfeature.LAND, facecolor='white', zorder=0)

# 添加海岸线，并设置其在陆地上方
ax.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=1) # <--- 设置 zorder

# 绘制散点数据，并设置其为最顶层
scatter = ax.scatter(
    map_df['Lon'], map_df['Lat'],
    c=map_df['AGB'],
    cmap='YlGn',
    s=5,
    edgecolor='none',
    transform=ccrs.PlateCarree(),
    vmin=0, vmax=300,
    zorder=2 # <--- 设置 zorder，确保点在最上面
)

# 创建内嵌的颜色条
cbar_ax = fig.add_axes([0.15, 0.25, 0.18, 0.06]) 
cbar = plt.colorbar(scatter, cax=cbar_ax, orientation='horizontal')
cbar.ax.set_title('红树林AGB密度 (Mg DM ha$^{-1}$)', fontsize=14, pad=10)
cbar.set_ticks([0, 100, 200, 300])
cbar.ax.tick_params(labelsize=12)

plt.show()